In [9]:
%matplotlib inline
import brian2 as b2
import matplotlib.pyplot as plt
import numpy as np
#from neurodynex3.leaky_integrate_and_fire import LIF
#rom neurodynex3.tools import input_factory, plot_tools


# parameters in SI units: volts, seconds, ohms, amps
V_rest  = -0.070    # -70 mV
V_reset = -0.065    # -65 mV
V_th    = -0.050    # -50 mV  firing threshold
R       = 10e6      # 10 MΩ
tau     = 8e-3      # 8 ms  membrane time constant
t_refr  = 2e-3      # 2 ms  absolute refractory period
dt = 1e-4;
rng = np.random.default_rng(0)

tau_syn = 5e-3   # 5 ms
gbar    = 40e-12 # 40 pS


def simulate_lif(I, dt, V_rest,V_reset,V_th, R, tau, t_refr):
    #simulate LIF model through differential equation itself
    n = len(I)
    V=np.zeros(n)
    V[0] = V_rest
    spike_times = []
    refr_steps = int(round(t_refr / dt))   # how many steps to stay clamped
    refr_counter = 0                       # >0 means we're inside refractory

    for k in range(n-1):
        if refr_counter>0:
            V[k+1] = V_reset
            refr_counter -= 1
            continue 
        #differential equation
        dV = (dt / tau) * (-(V[k] - V_rest) + R * I[k])
        V[k+1] = V[k] + dV

        # record spike timings 

        if V[k] < V_th and V[k+1] >= V_th:
            spike_times.append((k+1) * dt)
            V[k+1] = V_reset #reset
            refr_counter = refr_steps
    t = np.arange(n)* dt 
    return t, V, spike_times


def poisson_encode(rate, T, dt, rng):
    n = int(round(T / dt))
    spikes = (np.random.rand(n) < rate * dt).astype(float) #IDENTIFY how many spikes actually occur per timestep
    return spikes

def conductance_trace(spk, gbar, tau_syn, dt):
    #function to determine change in conductance in postsynaptic neuron
    n = len(spk)
    g = np.zeros(n)
    decay = np.exp(-dt / tau_syn)
    for k in range(1, n):
        g[k] = g[k-1] * decay + gbar * spk[k]
    return g 

In [7]:
#testing spike encoding
spk = poisson_encode(100, 0.5, 1e-4, rng)
print(spk.sum(), "spikes in 0.5s at 100Hz — expected ~50")

47.0 spikes in 0.5s at 100Hz — expected ~50


In [10]:
#testing conductance trace

g = conductance_trace(spk, gbar, tau_syn, dt)
print(f"peak conductance: {g.max()*1e12:.1f} pS")

peak conductance: 97.2 pS
